In [ ]:
!git clone https://github.com/liturriago/CrowdSegmenter-CT-GCE.git
%cd CrowdSegmenter-CT-GCE
!pip install -e ".[training,tuning,viz]" -q

In [ ]:
import kagglehub

path = kagglehub.dataset_download("lucasiturriago/retinal-fundus-glaucoma-analysis")
print("Path to dataset files:", path)

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

In [ ]:
import shlex

# Pon en None cualquier modelo DL que no quieras evaluar (basta con 1)
CTGCE_WEIGHTS = "/content/drive/MyDrive/CrowdSegmenter-CT-GCE/results/ctgce/riga/ct_gce_final.pth"
TGCE_SSPS_WEIGHTS = "/content/drive/MyDrive/CrowdSegmenter-CT-GCE/results/tgce_ssps/riga/tgce_ssps_final.pth"
CROWDSEG_WEIGHTS = "/content/drive/MyDrive/CrowdSegmenter-CT-GCE/results/crowdseg/riga/crowdseg_final.pth"

INCLUDE_MV = True
INCLUDE_STAPLE = True
METHODS = None  # Opcional, ej: ["CT-GCE", "CrowdSeg", "MV", "Spatial-STAPLE"]
REFERENCE = None  # Opcional (por defecto CT-GCE si está activo)
OUTPUT_DIR = "/content/drive/MyDrive/CrowdSegmenter-CT-GCE/results/benchmark/riga"

extra_flags = []
for flag, val in (
    ("--ctgce_weights", CTGCE_WEIGHTS),
    ("--tgce_ssps_weights", TGCE_SSPS_WEIGHTS),
    ("--crowdseg_weights", CROWDSEG_WEIGHTS),
):
    if val:
        extra_flags.extend([flag, shlex.quote(val)])
if not INCLUDE_MV:
    extra_flags.append("--no-mv")
if not INCLUDE_STAPLE:
    extra_flags.append("--no-staple")
if METHODS:
    extra_flags.append("--methods")
    extra_flags.extend(shlex.quote(m) for m in METHODS)
if REFERENCE:
    extra_flags.extend(["--reference", shlex.quote(REFERENCE)])
extra_cli = " ".join(extra_flags)

!python scripts/evaluate_benchmark.py --dataset riga --data "{path}" --output_dir "{OUTPUT_DIR}" --eval-nlc {extra_cli}